## 1. Installation et Imports

In [ ]:
# Imports
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configuration des graphiques
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
%matplotlib inline

# Ajouter le dossier src au path
sys.path.append('../src')

# Importer nos modules
from data_loader import (
    load_data, 
    handle_missing_values, 
    analyze_missing_values,
    get_data_summary,
    create_sample_dataset
)

from preprocessor import (
    DataPreprocessor,
    preprocess_data,
    split_train_test,
    create_feature_summary
)

from anomaly_detector import (
    IsolationForestDetector,
    OneClassSVMDetector,
    AutoencoderDetector,
    compare_detectors
)

from evaluator import (
    evaluate_predictions,
    evaluate_and_visualize,
    plot_confusion_matrix,
    plot_anomaly_scores,
    plot_scatter_2d,
    plot_feature_boxplots,
    generate_anomaly_report,
    compare_models_visualization
)

print("✅ Imports réussis !")

## 2. Chargement des Données

Pour cette démonstration, nous allons créer un dataset synthétique contenant des données normales et des anomalies.

**Note** : Vous pouvez remplacer cette étape par le chargement de vos propres données avec `load_data('chemin/vers/fichier.csv')`

In [ ]:
# Créer un dataset synthétique
# 1000 échantillons, 5 features numériques, 10% d'anomalies
df = create_sample_dataset(
    n_samples=1000,
    n_features=5,
    contamination=0.1,
    random_state=42
)

print(f"Dataset créé : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head(10)

## 3. Exploration des Données

Analysons la structure et les caractéristiques de nos données.

In [ ]:
# Résumé statistique
summary = get_data_summary(df)
print("\n" + "="*60)
print("STATISTIQUES DU DATASET")
print("="*60)

In [ ]:
# Statistiques descriptives
df.describe()

In [ ]:
# Résumé des features
feature_summary = create_feature_summary(df)
feature_summary

In [ ]:
# Analyse des valeurs manquantes
missing_analysis = analyze_missing_values(df)

In [ ]:
# Visualisation de la distribution des vraies étiquettes
true_labels = df['true_label'].value_counts()
print(f"\nDistribution des vraies étiquettes :")
print(f"Normal (0) : {true_labels[0]} ({true_labels[0]/len(df)*100:.2f}%)")
print(f"Anomalie (1) : {true_labels[1]} ({true_labels[1]/len(df)*100:.2f}%)")

plt.figure(figsize=(8, 5))
true_labels.plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Distribution des Vraies Étiquettes')
plt.xlabel('Classe (0=Normal, 1=Anomalie)')
plt.ylabel('Nombre d\'échantillons')
plt.xticks(rotation=0)
plt.show()

## 4. Prétraitement des Données

### 4.1 Traitement des Valeurs Manquantes

In [ ]:
# Gérer les valeurs manquantes
df_cleaned = handle_missing_values(
    df,
    strategy='auto',
    threshold=0.5,
    numeric_method='mean',
    categorical_method='mode'
)

print(f"\nFormes avant/après nettoyage : {df.shape} -> {df_cleaned.shape}")

### 4.2 Normalisation et Encodage

In [ ]:
# Prétraiter les données (exclure 'id' et 'true_label')
X, preprocessor = preprocess_data(
    df_cleaned,
    numeric_scaling='standard',  # Standardisation (mean=0, std=1)
    categorical_encoding='onehot',  # One-hot encoding pour les catégories
    exclude_columns=['id', 'true_label'],
    return_preprocessor=True
)

print(f"\nForme des données transformées : {X.shape}")
print(f"Noms des features : {preprocessor.get_feature_names()}")

In [ ]:
# Extraire les vraies étiquettes (pour l'évaluation)
y_true = df_cleaned['true_label'].values
print(f"Vraies étiquettes : {len(y_true)} échantillons")

## 5. Détection d'Anomalies

Nous allons maintenant appliquer les trois algorithmes de détection d'anomalies.

### 5.1 Isolation Forest

In [ ]:
# Initialiser et entraîner Isolation Forest
if_detector = IsolationForestDetector(
    contamination=0.1,  # Proportion attendue d'anomalies
    n_estimators=100,
    random_state=42
)

# Entraîner et prédire
if_predictions = if_detector.fit_predict(X)
if_scores = if_detector.get_anomaly_scores(X)

print(f"\nAnomalies détectées : {np.sum(if_predictions == -1)}/{len(if_predictions)}")

In [ ]:
# Évaluer Isolation Forest
if_metrics = evaluate_predictions(y_true, if_predictions, "Isolation Forest")

### 5.2 One-Class SVM

In [ ]:
# Initialiser et entraîner One-Class SVM
svm_detector = OneClassSVMDetector(
    nu=0.1,  # Similaire à contamination
    kernel='rbf',
    gamma='scale'
)

# Entraîner et prédire
svm_predictions = svm_detector.fit_predict(X)
svm_scores = svm_detector.get_anomaly_scores(X)

print(f"\nAnomalies détectées : {np.sum(svm_predictions == -1)}/{len(svm_predictions)}")

In [ ]:
# Évaluer One-Class SVM
svm_metrics = evaluate_predictions(y_true, svm_predictions, "One-Class SVM")

### 5.3 Autoencodeur

In [ ]:
# Initialiser et entraîner l'Autoencodeur
ae_detector = AutoencoderDetector(
    encoding_dim=8,
    hidden_layers=None,  # Sera défini automatiquement
    epochs=50,
    batch_size=32,
    contamination=0.1,
    verbose=1,  # Afficher la progression
    random_state=42
)

# Entraîner et prédire
ae_predictions = ae_detector.fit_predict(X)
ae_scores = ae_detector.get_anomaly_scores(X)

print(f"\nAnomalies détectées : {np.sum(ae_predictions == -1)}/{len(ae_predictions)}")

In [ ]:
# Évaluer l'Autoencodeur
ae_metrics = evaluate_predictions(y_true, ae_predictions, "Autoencodeur")

## 6. Comparaison des Modèles

### 6.1 Tableau Comparatif

In [ ]:
# Créer un tableau comparatif des métriques
comparison_df = pd.DataFrame([
    {
        'Modèle': 'Isolation Forest',
        'Précision': if_metrics['precision'],
        'Rappel': if_metrics['recall'],
        'F1-Score': if_metrics['f1_score'],
        'Anomalies Détectées': if_metrics['n_anomalies_pred']
    },
    {
        'Modèle': 'One-Class SVM',
        'Précision': svm_metrics['precision'],
        'Rappel': svm_metrics['recall'],
        'F1-Score': svm_metrics['f1_score'],
        'Anomalies Détectées': svm_metrics['n_anomalies_pred']
    },
    {
        'Modèle': 'Autoencodeur',
        'Précision': ae_metrics['precision'],
        'Rappel': ae_metrics['recall'],
        'F1-Score': ae_metrics['f1_score'],
        'Anomalies Détectées': ae_metrics['n_anomalies_pred']
    }
])

print("\n" + "="*80)
print("COMPARAISON DES MODÈLES")
print("="*80)
print(comparison_df.to_string(index=False))
print(f"\nNombre réel d'anomalies : {if_metrics['n_anomalies_true']}")

In [ ]:
# Visualiser la comparaison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_to_plot = ['Précision', 'Rappel', 'F1-Score']
colors = ['#3498db', '#e74c3c', '#2ecc71']

for i, metric in enumerate(metrics_to_plot):
    axes[i].bar(comparison_df['Modèle'], comparison_df[metric], color=colors)
    axes[i].set_title(f'Comparaison : {metric}', fontsize=14, fontweight='bold')
    axes[i].set_ylabel(metric, fontsize=12)
    axes[i].set_ylim([0, 1])
    axes[i].tick_params(axis='x', rotation=45)
    
    # Ajouter les valeurs sur les barres
    for j, v in enumerate(comparison_df[metric]):
        axes[i].text(j, v + 0.02, f'{v:.3f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

### 6.2 Matrices de Confusion

In [ ]:
# Matrices de confusion pour chaque modèle
plot_confusion_matrix(y_true, if_predictions, "Isolation Forest")

In [ ]:
plot_confusion_matrix(y_true, svm_predictions, "One-Class SVM")

In [ ]:
plot_confusion_matrix(y_true, ae_predictions, "Autoencodeur")

## 7. Visualisation des Résultats

### 7.1 Distribution des Scores d'Anomalie

In [ ]:
# Isolation Forest
plot_anomaly_scores(if_scores, if_predictions, y_true, "Isolation Forest")

In [ ]:
# One-Class SVM
plot_anomaly_scores(svm_scores, svm_predictions, y_true, "One-Class SVM")

In [ ]:
# Autoencodeur
plot_anomaly_scores(ae_scores, ae_predictions, y_true, "Autoencodeur")

### 7.2 Visualisation 2D des Anomalies

In [ ]:
# Comparaison visuelle des trois modèles
results_dict = {
    'Isolation Forest': if_predictions,
    'One-Class SVM': svm_predictions,
    'Autoencodeur': ae_predictions
}

compare_models_visualization(
    X,
    results_dict,
    y_true=y_true,
    feature_names=preprocessor.get_feature_names(),
    features_to_plot=(0, 1)  # Afficher les 2 premières features
)

### 7.3 Boxplots des Features

In [ ]:
# Boxplots pour Isolation Forest
numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in ['id', 'true_label']]

plot_feature_boxplots(
    df_cleaned,
    if_predictions,
    numeric_columns=numeric_cols,
    max_features=6
)

## 8. Génération de Rapports d'Anomalies

### 8.1 Top Anomalies - Isolation Forest

In [ ]:
# Générer un rapport des top anomalies
if_report = generate_anomaly_report(
    df_cleaned,
    if_predictions,
    if_scores,
    top_n=20
)

if_report.head(10)

### 8.2 Top Anomalies - One-Class SVM

In [ ]:
svm_report = generate_anomaly_report(
    df_cleaned,
    svm_predictions,
    svm_scores,
    top_n=20
)

svm_report.head(10)

### 8.3 Top Anomalies - Autoencodeur

In [ ]:
ae_report = generate_anomaly_report(
    df_cleaned,
    ae_predictions,
    ae_scores,
    top_n=20
)

ae_report.head(10)

### 8.4 Sauvegarder les Rapports

In [ ]:
# Créer un dossier pour les résultats
os.makedirs('../results', exist_ok=True)

# Sauvegarder les rapports
if_report.to_csv('../results/anomalies_isolation_forest.csv', index=False)
svm_report.to_csv('../results/anomalies_onesvm.csv', index=False)
ae_report.to_csv('../results/anomalies_autoencoder.csv', index=False)

print("✅ Rapports sauvegardés dans le dossier 'results/'")

## 9. Sauvegarder les Modèles

In [ ]:
# Sauvegarder les modèles entraînés
if_detector.save('../models/isolation_forest_model.pkl')
svm_detector.save('../models/onesvm_model.pkl')
ae_detector.save('../models/autoencoder_model.pkl')

# Sauvegarder le préprocesseur
preprocessor.save('../models/preprocessor.pkl')

print("✅ Modèles et préprocesseur sauvegardés dans le dossier 'models/'")

## 10. Conclusions et Recommandations

### Résumé des Résultats

Basé sur notre analyse, voici ce que nous avons observé :

1. **Isolation Forest** :
   - ✅ Rapide et efficace
   - ✅ Fonctionne bien sur des datasets de taille moyenne à grande
   - ✅ Peu de paramètres à ajuster
   - ⚠️ Peut avoir du mal avec des anomalies subtiles

2. **One-Class SVM** :
   - ✅ Performant pour des frontières complexes
   - ✅ Robuste mathématiquement
   - ⚠️ Plus lent sur de grands datasets
   - ⚠️ Nécessite un ajustement des hyperparamètres

3. **Autoencodeur** :
   - ✅ Excellent pour des patterns complexes
   - ✅ Peut apprendre des représentations non-linéaires
   - ⚠️ Nécessite plus de données d'entraînement
   - ⚠️ Plus long à entraîner

### Recommandations

**Choisir le bon algorithme selon votre cas d'usage :**

- **Isolation Forest** : Premier choix pour la plupart des cas, rapide et efficace
- **One-Class SVM** : Quand vous avez besoin de frontières de décision complexes
- **Autoencodeur** : Pour des patterns très complexes avec suffisamment de données

**Conseils pratiques :**

1. Toujours commencer par une exploration approfondie des données
2. Tester plusieurs algorithmes et comparer les résultats
3. Ajuster le paramètre `contamination` selon votre domaine
4. Valider manuellement un échantillon des anomalies détectées
5. Réentraîner régulièrement les modèles avec de nouvelles données

### Prochaines Étapes

1. Appliquer ces techniques sur vos propres données
2. Ajuster les hyperparamètres pour optimiser les performances
3. Intégrer la détection d'anomalies dans votre pipeline de production
4. Mettre en place un système de monitoring et d'alerte

---

**📊 Merci d'avoir suivi ce notebook !**